# Feature Definitions — Two-Tower Recommender

Defines all Databricks Feature Views for the real-time recommender: session-level streaming features (StreamSource + RollingWindow), product batch features (DeltaTableSource + ColumnSelection), product rolling aggregates (DeltaTableSource + SlidingWindow), and request-time features (RequestSource).

In [0]:
%pip install databricks-feature-engineering>=0.16.0
dbutils.library.restartPython()

In [0]:
import json
from datetime import timedelta
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering.entities import (
    DeltaTableSource, StreamSource, Feature, AggregationFunction,
    Count, LastDistinct, Max, Min, SlidingWindow, RollingWindow, Sum, Avg,
    StddevPop, RequestSource, FieldDefinition, ScalarDataType, ColumnSelection,
)

fe = FeatureEngineeringClient()

In [0]:
import yaml
import os

config_path = os.path.join(os.getcwd(), "config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

CATALOG = config["catalog"]
SCHEMA = config["schema"]
PRODUCT_TABLE = config["product_table"]
PRODUCT_AGGREGATES_TABLE = config["product_aggregates_table"]
STREAM_NAME = config["stream_name"]
STREAM_SOURCE = f"{CATALOG}.{SCHEMA}.{STREAM_NAME}"

print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")
print(f"Product table:          {PRODUCT_TABLE}")
print(f"Product aggregates:     {PRODUCT_AGGREGATES_TABLE}")
print(f"Stream source:          {STREAM_SOURCE}")

In [0]:
# --- Single StreamSource with transformation_sql ---
# Derives indicator columns for event-type counts and cart value,
# enabling all session features from one source (one materialization pipeline).
# Sessions with no events of a given type get 0 (not null) for counts and values.
PROJECTED_SCHEMA_JSON = json.dumps({
    "type": "struct",
    "fields": [
        {"name": "user_session", "type": "string", "nullable": True, "metadata": {}},
        {"name": "event_time", "type": "timestamp", "nullable": True, "metadata": {}},
        {"name": "event_type", "type": "string", "nullable": True, "metadata": {}},
        {"name": "product_id", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "brand", "type": "string", "nullable": True, "metadata": {}},
        {"name": "price", "type": "double", "nullable": True, "metadata": {}},
        {"name": "is_view", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "is_cart", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "is_purchase", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "cart_price", "type": "double", "nullable": True, "metadata": {}},
    ]
})

session_source = StreamSource(
    full_name=STREAM_SOURCE,
    transformation_sql=(
        "value.user_session AS user_session, "
        "value.event_time AS event_time, "
        "value.event_type AS event_type, "
        "value.product_id AS product_id, "
        "value.brand AS brand, "
        "value.price AS price, "
        "CASE WHEN value.event_type = 'view' THEN 1 ELSE 0 END AS is_view, "
        "CASE WHEN value.event_type = 'cart' THEN 1 ELSE 0 END AS is_cart, "
        "CASE WHEN value.event_type = 'purchase' THEN 1 ELSE 0 END AS is_purchase, "
        "CASE WHEN value.event_type = 'cart' THEN value.price ELSE 0.0 END AS cart_price"
    ),
    dataframe_schema=PROJECTED_SCHEMA_JSON,
)

SESSION_WINDOW = RollingWindow(
    window_duration=timedelta(hours=24),
)

session_event_count = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Count(input="event_type"), SESSION_WINDOW),
    name="session_event_count",
)

session_avg_price = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Avg(input="price"), SESSION_WINDOW),
    name="session_avg_price",
)

session_price_stddev = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(StddevPop(input="price"), SESSION_WINDOW),
    name="session_price_stddev",
)

session_total_value = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="price"), SESSION_WINDOW),
    name="session_total_value",
)

session_recent_products = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(LastDistinct(input="product_id", n=10), SESSION_WINDOW),
    name="session_recent_products",
)

session_view_count = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_view"), SESSION_WINDOW),
    name="session_view_count",
)

session_cart_count = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_cart"), SESSION_WINDOW),
    name="session_cart_count",
)

session_purchase_count = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_purchase"), SESSION_WINDOW),
    name="session_purchase_count",
)

session_cart_value = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="cart_price"), SESSION_WINDOW),
    name="session_cart_value",
)

session_recent_brands = Feature(
    source=session_source,
    entity=["user_session"],
    timeseries_column="event_time",
    function=AggregationFunction(LastDistinct(input="brand", n=5), SESSION_WINDOW),
    name="session_recent_brands",
)

session_features = [
    session_event_count,
    session_avg_price,
    session_price_stddev,
    session_total_value,
    session_recent_products,
    session_view_count,
    session_cart_count,
    session_purchase_count,
    session_cart_value,
    session_recent_brands,
]

In [0]:
product_source = DeltaTableSource(
    catalog_name=CATALOG,
    schema_name=SCHEMA,
    table_name=PRODUCT_TABLE,
)

# --- Item price (raw) ---
item_price = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("price"),
    name="item_price",
)

# --- Item category L1 (top-level) ---
item_category_l1 = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("category_l1"),
    name="item_category_l1",
)

# --- Item category L2 (mid-level) ---
item_category_l2 = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("category_l2"),
    name="item_category_l2",
)

# --- Item category L3 (leaf) ---
item_category_l3 = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("category_l3"),
    name="item_category_l3",
)

# --- Item brand ---
item_brand = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("brand"),
    name="item_brand",
)

# --- Item log price (log-transformed for model input) ---
item_log_price = Feature(
    source=product_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=ColumnSelection("log_price"),
    name="item_log_price",
)

product_batch_features = [
    item_price,
    item_category_l1,
    item_category_l2,
    item_category_l3,
    item_brand,
    item_log_price,
]

In [0]:
# --- Single DeltaTableSource with transformation_sql ---
# Derives indicator columns for event-type counts, consolidating four sources into one.
# Products with no events of a given type get 0 (not null) for counts.
PRODUCT_AGG_SCHEMA_JSON = json.dumps({
    "type": "struct",
    "fields": [
        {"name": "product_id", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "event_time", "type": "timestamp", "nullable": True, "metadata": {}},
        {"name": "event_time_copy", "type": "timestamp", "nullable": True, "metadata": {}},
        {"name": "is_view", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "is_cart", "type": "integer", "nullable": True, "metadata": {}},
        {"name": "is_purchase", "type": "integer", "nullable": True, "metadata": {}},
    ]
})

clickstream_source = DeltaTableSource(
    catalog_name=CATALOG,
    schema_name=SCHEMA,
    table_name=PRODUCT_AGGREGATES_TABLE,
    transformation_sql=(
        "product_id AS product_id, "
        "event_time AS event_time, "
        "event_time_copy AS event_time_copy, "
        "CASE WHEN event_type = 'view' THEN 1 ELSE 0 END AS is_view, "
        "CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END AS is_cart, "
        "CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END AS is_purchase"
    ),
    dataframe_schema=PRODUCT_AGG_SCHEMA_JSON,
)

PRODUCT_AGG_WINDOW = SlidingWindow(
    window_duration=timedelta(days=7),
    slide_duration=timedelta(days=1),
)

# --- Item view count (7-day popularity) ---
item_view_count = Feature(
    source=clickstream_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_view"), PRODUCT_AGG_WINDOW),
    name="item_view_count",
)

# --- Item cart count (7-day purchase intent) ---
item_cart_count = Feature(
    source=clickstream_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_cart"), PRODUCT_AGG_WINDOW),
    name="item_cart_count",
)

# --- Item purchase count (7-day conversion) ---
item_purchase_count = Feature(
    source=clickstream_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=AggregationFunction(Sum(input="is_purchase"), PRODUCT_AGG_WINDOW),
    name="item_purchase_count",
)

# --- Item recency (most recent interaction) ---
# Max(event_time) fails because event_time is the timeseries_column; use event_time_copy.
item_recency = Feature(
    source=clickstream_source,
    entity=["product_id"],
    timeseries_column="event_time",
    function=AggregationFunction(Max(input="event_time_copy"), PRODUCT_AGG_WINDOW),
    name="item_recency",
)

# --- Category price bounds (from product catalog, not clickstream) ---
category_l1_min_price = Feature(
    source=product_source,
    entity=["category_l1"],
    timeseries_column="event_time",
    function=AggregationFunction(Min(input="price"), PRODUCT_AGG_WINDOW),
    name="category_l1_min_price",
)

category_l1_max_price = Feature(
    source=product_source,
    entity=["category_l1"],
    timeseries_column="event_time",
    function=AggregationFunction(Max(input="price"), PRODUCT_AGG_WINDOW),
    name="category_l1_max_price",
)

category_l2_min_price = Feature(
    source=product_source,
    entity=["category_l2"],
    timeseries_column="event_time",
    function=AggregationFunction(Min(input="price"), PRODUCT_AGG_WINDOW),
    name="category_l2_min_price",
)

category_l2_max_price = Feature(
    source=product_source,
    entity=["category_l2"],
    timeseries_column="event_time",
    function=AggregationFunction(Max(input="price"), PRODUCT_AGG_WINDOW),
    name="category_l2_max_price",
)

product_rolling_features = [
    item_view_count,
    item_cart_count,
    item_purchase_count,
    item_recency,
    category_l1_min_price,
    category_l1_max_price,
    category_l2_min_price,
    category_l2_max_price,
]

In [0]:
request_source = RequestSource(
    schema=[
        FieldDefinition(name="product_id", data_type=ScalarDataType.INTEGER),
        FieldDefinition(name="category_l1", data_type=ScalarDataType.STRING),
        FieldDefinition(name="category_l2", data_type=ScalarDataType.STRING),
        FieldDefinition(name="hour_of_day", data_type=ScalarDataType.INTEGER),
        FieldDefinition(name="day_of_week", data_type=ScalarDataType.INTEGER),
    ]
)

product_id_feature = Feature(source=request_source, function=ColumnSelection("product_id"), name="product_id")
category_l1_feature = Feature(source=request_source, function=ColumnSelection("category_l1"), name="category_l1")
category_l2_feature = Feature(source=request_source, function=ColumnSelection("category_l2"), name="category_l2")
hour_of_day_feature = Feature(source=request_source, function=ColumnSelection("hour_of_day"), name="hour_of_day")
day_of_week_feature = Feature(source=request_source, function=ColumnSelection("day_of_week"), name="day_of_week")

request_features = [
    product_id_feature, category_l1_feature, category_l2_feature,
    hour_of_day_feature, day_of_week_feature,
]

In [0]:
all_features = session_features + product_batch_features + product_rolling_features + request_features
print(f"Total features defined: {len(all_features)}")
print(f"  Session (StreamSource + RollingWindow):    {len(session_features)}")
print(f"  Product batch (ColumnSelection):           {len(product_batch_features)}")
print(f"  Product rolling (SlidingWindow):           {len(product_rolling_features)}")
print(f"  Request (RequestSource):                   {len(request_features)}")

In [0]:
from pyspark.sql.window import Window

# --- Source table and output tables ---
TABLE = f"{CATALOG}.{SCHEMA}.{config['ecommerce_table']}"
PRODUCT_TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{config['product_train_table']}"
SESSION_TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.{config['session_train_table']}"
EVALUATION_TABLE = f"{CATALOG}.{SCHEMA}.{config['eval_table']}"

# Shift 2019 clickstream dates to 2026 for Feature View PIT joins
DELTA_DAYS = config["delta_days"]

# --- Per-tower feature splits (for fe.log_model lineage) ---
# Item tower: product-level features + product_id entity key
item_tower_features = product_batch_features + product_rolling_features + [product_id_feature]

# Session tower: session streaming features + session-level request features
session_tower_features = session_features + [
    category_l1_feature,
    category_l2_feature,
    hour_of_day_feature,
    day_of_week_feature,
]

print(f"Item tower:    {len(item_tower_features)} features")
print(f"Session tower: {len(session_tower_features)} features")
print(f"Combined:      {len(all_features)} features")

In [0]:
# ---------------------------------------------------------------------------
# Build labeled DataFrames from raw clickstream, join all features via
# create_training_set (point-in-time correct), and persist train/eval tables.
# ---------------------------------------------------------------------------

full_df = spark.table(TABLE)

w_session = Window.partitionBy("user_session")

labeled_df = (
    full_df
    .filter(F.to_date("event_time").between(config["training_start_day"], config["evaluation_end_day"]))
    .withColumn("category_l1", F.coalesce(F.split("category_code", "\\.").getItem(0), F.lit("unknown")))
    .withColumn("category_l2", F.coalesce(F.split("category_code", "\\.").getItem(1), F.lit("unknown")))
    .select(
        "product_id", "category_l1", "category_l2",
        "user_session", "event_time", "event_type", "price",
    )
    .withColumn("session_first_date", F.min(F.to_date("event_time")).over(w_session))
    .withColumn("relevance",
        F.when(F.col("event_type") == "purchase", 3)
         .when(F.col("event_type") == "cart", 2)
         .otherwise(1))
    .withColumn("hour_of_day", F.hour("event_time"))
    .withColumn("day_of_week", F.dayofweek("event_time") - 1)
)

# Train/eval split from config (using original 2019 dates for filtering)
labeled_train = labeled_df.filter(F.col("session_first_date") <= config["training_end_day"])
labeled_eval = labeled_df.filter(F.col("session_first_date") >= config["evaluation_start_day"])

# Shift event_time forward to align with Feature View source tables.
# The product + aggregates tables were created with DELTA_DAYS=2491 shift
# (notebook 01), so their event_time is in 2026. The raw clickstream has
# original 2019 dates. Without this shift, point-in-time Feature View
# joins return NULL for all product features.
labeled_train = labeled_train.withColumn("event_time", F.expr(f"event_time + INTERVAL {DELTA_DAYS} DAYS"))
labeled_eval = labeled_eval.withColumn("event_time", F.expr(f"event_time + INTERVAL {DELTA_DAYS} DAYS"))

# --- Per-tower training sets (objects for fe.log_model feature lineage) ---
item_training_set = fe.create_training_set(
    df=labeled_train, features=item_tower_features,
    label="event_type", exclude_columns=["session_first_date", "price"],
)
session_training_set = fe.create_training_set(
    df=labeled_train, features=session_tower_features,
    label="event_type", exclude_columns=["session_first_date", "price"],
)

# --- Combined training set (all features, for training loop) ---
training_set = fe.create_training_set(
    df=labeled_train, features=all_features,
    label="event_type", exclude_columns=["session_first_date", "price"],
)
training_df = training_set.load_df()

# --- Split into per-tower tables ---
_SESSION_FV_COLS = {
    "session_event_count", "session_avg_price", "session_price_stddev", "session_total_value",
    "session_recent_products", "session_view_count", "session_cart_count",
    "session_purchase_count", "session_cart_value", "session_recent_brands",
}
_ITEM_FV_COLS = {
    "item_price", "item_category_l1", "item_category_l2", "item_category_l3",
    "item_brand", "item_log_price", "item_view_count", "item_cart_count",
    "item_purchase_count", "item_recency", "category_l1_min_price",
    "category_l1_max_price", "category_l2_min_price", "category_l2_max_price",
}
product_cols = [c for c in training_df.columns if c not in _SESSION_FV_COLS]
session_cols = [c for c in training_df.columns if c not in _ITEM_FV_COLS]

training_df.select(product_cols).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(PRODUCT_TRAIN_TABLE)
training_df.select(session_cols).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SESSION_TRAIN_TABLE)

# --- Eval set (all features) ---
eval_set = fe.create_training_set(
    df=labeled_eval, features=all_features,
    label="event_type", exclude_columns=["session_first_date", "price"],
)
eval_df = eval_set.load_df()
eval_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(EVALUATION_TABLE)

# For downstream use
train_df = training_df
eval_df = spark.table(EVALUATION_TABLE)

product_count = spark.table(PRODUCT_TRAIN_TABLE).count()
session_count = spark.table(SESSION_TRAIN_TABLE).count()
eval_count = spark.table(EVALUATION_TABLE).count()
print(f"Product train: {product_count:,} rows \u2192 {PRODUCT_TRAIN_TABLE}")
print(f"Session train: {session_count:,} rows \u2192 {SESSION_TRAIN_TABLE}")
print(f"Eval set:      {eval_count:,} rows \u2192 {EVALUATION_TABLE}")

In [0]:
# ---------------------------------------------------------------------------
# Nov 4-6 held-out test set.
# Same pipeline as cell 11 (labeled DF + fe.create_training_set), but
# restricted to sessions whose first event falls in Nov 4-6 2019.
# Writes to a SEPARATE table so the eval table is untouched.
# ---------------------------------------------------------------------------
TEST_TABLE = f"{CATALOG}.{SCHEMA}.{config['test_table']}"

full_df = spark.table(TABLE)
w_session = Window.partitionBy("user_session")

# Include the full date range so that sessions starting before Nov 4
# that have prior events are captured for Feature View PIT joins.
labeled_df = (
    full_df
    .filter(F.to_date("event_time").between(config["training_start_day"], config["test_end_day"]))
    .withColumn("category_l1", F.coalesce(F.split("category_code", "\\.").getItem(0), F.lit("unknown")))
    .withColumn("category_l2", F.coalesce(F.split("category_code", "\\.").getItem(1), F.lit("unknown")))
    .select(
        "product_id", "category_l1", "category_l2",
        "user_session", "event_time", "event_type", "price",
    )
    .withColumn("session_first_date", F.min(F.to_date("event_time")).over(w_session))
    .withColumn("relevance",
        F.when(F.col("event_type") == "purchase", 3)
         .when(F.col("event_type") == "cart", 2)
         .otherwise(1))
    .withColumn("hour_of_day", F.hour("event_time"))
    .withColumn("day_of_week", F.dayofweek("event_time") - 1)
)

# Test split: sessions whose first event falls in Nov 4-6
labeled_test = labeled_df.filter(
    (F.col("session_first_date") >= "2019-11-04") &
    (F.col("session_first_date") <= "2019-11-06")
)

# Shift event_time to align with Feature View source tables (shifted by 2491 days)
labeled_test = labeled_test.withColumn("event_time", F.expr(f"event_time + INTERVAL {DELTA_DAYS} DAYS"))

print(f"Test rows (Nov 4-6 sessions): {labeled_test.count():,}")

# Create test set with all features (point-in-time correct)
test_set = fe.create_training_set(
    df=labeled_test, features=all_features,
    label="event_type", exclude_columns=["session_first_date", "price"],
)
test_df = test_set.load_df()
test_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TEST_TABLE)

test_count = spark.table(TEST_TABLE).count()
print(f"Test set: {test_count:,} rows \u2192 {TEST_TABLE}")